# Geospatial Analysis - Nearest Warehouse Assignment

This notebook solves the logistics problem of assigning 50 deliveries to the nearest warehouse using the Haversine distance formula.

## Problem Setup
- **3 Warehouses**: Delhi, Mumbai, Chennai
- **50 Deliveries**: Each with latitude/longitude coordinates
- **Goal**: Assign each delivery to the nearest warehouse and find which warehouse handles the most deliveries

## Step 1: Upload Your Data File

Upload the `deliveries.csv` file using the file upload button below:

In [ ]:
from google.colab import files
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# Upload the CSV file
uploaded = files.upload()
print("\n✅ File uploaded successfully!")

## Step 2: Define the Haversine Distance Function

The Haversine formula calculates the great-circle distance between two points on a sphere (Earth) given their longitudes and latitudes.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    Returns distance in kilometers
    """
    R = 6371  # Radius of Earth in kilometers
    
    # Convert decimal degrees to radians
    lat1_rad = radians(lat1)
    lon1_rad = radians(lon1)
    lat2_rad = radians(lat2)
    lon2_rad = radians(lon2)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = sin(dlat/2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    distance = R * c
    return distance

print("✅ Haversine function defined")

## Step 3: Define Warehouse Locations

In [ ]:
# Define warehouse locations
warehouses = {
    'Delhi': {'lat': 28.6139, 'lon': 77.209},
    'Mumbai': {'lat': 19.076, 'lon': 72.8777},
    'Chennai': {'lat': 13.0827, 'lon': 80.2707}
}

print("Warehouse Locations:")
print("=" * 50)
for name, coords in warehouses.items():
    print(f"{name:8s}: Lat {coords['lat']:8.4f}, Lon {coords['lon']:8.4f}")

## Step 4: Load and Explore the Data

In [ ]:
# Read the deliveries data
df = pd.read_csv('deliveries.csv')

print(f"Total deliveries to process: {len(df)}")
print("\nFirst 5 deliveries:")
print(df.head())
print("\nData Info:")
print(df.info())

## Step 5: Calculate Distances to Each Warehouse

In [ ]:
print("Calculating Haversine distances...\n")

# Calculate distance from each delivery to each warehouse
for warehouse_name, warehouse_coords in warehouses.items():
    distance_col = f'Distance_to_{warehouse_name}_km'
    df[distance_col] = df.apply(
        lambda row: haversine(
            row['Latitude'], 
            row['Longitude'],
            warehouse_coords['lat'],
            warehouse_coords['lon']
        ),
        axis=1
    )
    print(f"✓ Calculated distances to {warehouse_name}")

# Display sample with distances
print("\nSample data with distances (first 5 rows):")
print(df[['Delivery_ID', 'Latitude', 'Longitude', 'Distance_to_Delhi_km', 
          'Distance_to_Mumbai_km', 'Distance_to_Chennai_km']].head())

## Step 6: Assign Each Delivery to Nearest Warehouse

In [ ]:
# Find the nearest warehouse for each delivery
distance_columns = [f'Distance_to_{name}_km' for name in warehouses.keys()]

# Get the warehouse with minimum distance
df['Nearest_Warehouse'] = df[distance_columns].idxmin(axis=1)
df['Nearest_Warehouse'] = df['Nearest_Warehouse'].str.replace('Distance_to_', '').str.replace('_km', '')
df['Min_Distance_km'] = df[distance_columns].min(axis=1)

# Display sample assignments
print("Sample warehouse assignments (first 10):")
print("=" * 80)
print(df[['Delivery_ID', 'Latitude', 'Longitude', 'Nearest_Warehouse', 
          'Min_Distance_km']].head(10))

## Step 7: Count Deliveries Per Warehouse

In [ ]:
# Count deliveries per warehouse
warehouse_counts = df['Nearest_Warehouse'].value_counts().sort_index()

print("WAREHOUSE ASSIGNMENT SUMMARY")
print("=" * 60)
print("\nDeliveries per warehouse:")
for warehouse, count in warehouse_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {warehouse:8s}: {count:3d} deliveries ({percentage:5.1f}%)")

# Create a bar chart
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
warehouse_counts.plot(kind='bar', color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.title('Deliveries Assigned to Each Warehouse', fontsize=16, fontweight='bold')
plt.xlabel('Warehouse', fontsize=12)
plt.ylabel('Number of Deliveries', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8: Find the Busiest Warehouse (ANSWER)

In [ ]:
# Find the busiest warehouse
busiest_warehouse = warehouse_counts.idxmax()
max_deliveries = warehouse_counts.max()

print("=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"\n🏆 BUSIEST WAREHOUSE: {busiest_warehouse}")
print(f"📦 DELIVERY COUNT: {max_deliveries}")
print(f"\n" + "=" * 60)
print(f"✅ SUBMISSION FORMAT: {busiest_warehouse}, {max_deliveries}")
print("=" * 60)

## Step 9: Additional Statistics and Insights

In [ ]:
print("ADDITIONAL STATISTICS")
print("=" * 60)
print(f"\nAverage distance to nearest warehouse: {df['Min_Distance_km'].mean():.2f} km")
print(f"Maximum distance to nearest warehouse: {df['Min_Distance_km'].max():.2f} km")
print(f"Minimum distance to nearest warehouse: {df['Min_Distance_km'].min():.2f} km")

print("\n" + "=" * 60)
print("DISTANCE DISTRIBUTION BY WAREHOUSE")
print("=" * 60)

for warehouse in warehouses.keys():
    warehouse_df = df[df['Nearest_Warehouse'] == warehouse]
    if len(warehouse_df) > 0:
        avg_dist = warehouse_df['Min_Distance_km'].mean()
        print(f"\n{warehouse}:")
        print(f"  Deliveries: {len(warehouse_df)}")
        print(f"  Avg Distance: {avg_dist:.2f} km")
        print(f"  Distance Range: {warehouse_df['Min_Distance_km'].min():.2f} - {warehouse_df['Min_Distance_km'].max():.2f} km")

## Step 10: Save Results

In [ ]:
# Save detailed results to CSV
output_file = 'warehouse_assignments_detailed.csv'
df.to_csv(output_file, index=False)
print(f"💾 Detailed results saved to: {output_file}")

# Download the results file
files.download(output_file)
print("\n✅ File download started!")

## Visualization: Delivery Locations and Warehouses

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a scatter plot
fig, ax = plt.subplots(figsize=(14, 10))

# Color map for warehouses
colors = {'Delhi': '#FF6B6B', 'Mumbai': '#4ECDC4', 'Chennai': '#45B7D1'}

# Plot deliveries colored by assigned warehouse
for warehouse in warehouses.keys():
    warehouse_df = df[df['Nearest_Warehouse'] == warehouse]
    ax.scatter(warehouse_df['Longitude'], warehouse_df['Latitude'], 
               c=colors[warehouse], label=f'{warehouse} ({len(warehouse_df)} deliveries)',
               alpha=0.6, s=100, edgecolors='black', linewidth=0.5)

# Plot warehouses as stars
for warehouse, coords in warehouses.items():
    ax.scatter(coords['lon'], coords['lat'], 
               c=colors[warehouse], marker='*', s=1000, 
               edgecolors='black', linewidth=2, 
               label=f'{warehouse} Warehouse', zorder=5)

ax.set_xlabel('Longitude', fontsize=12, fontweight='bold')
ax.set_ylabel('Latitude', fontsize=12, fontweight='bold')
ax.set_title('Delivery Locations and Warehouse Assignments', fontsize=16, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()